# 00 — CNN image classification with TensorFlow/Keras

**Framework track:** TensorFlow/Keras only. Designed as a modern, explained companion to the original Colab CNN notebook. The original notebook remains at repository root and keeps its Colab badge.

## Learning objectives
- inspect a two-class image-folder dataset and verify labels;
- build a `tf.data` input pipeline;
- understand image normalization, augmentation and CNN dimensions;
- train with validation data and evaluate once on a held-out test set.

**Dataset:** Homer/Bart archive referenced by the original notebook as `Datasets/homer_bart_2.zip`. This archive is not committed in this repository. Supply your legally obtained local copy and verify its provenance/license.

## 1. Environment and version audit

Recommended modern baseline: TensorFlow 2.21.0 with its bundled Keras 3. Do not install PyTorch in this environment.

In [ ]:
import tensorflow as tf, numpy as np, pathlib, platform
print('TensorFlow:', tf.__version__)
print('Keras:', tf.keras.__version__ if hasattr(tf.keras, '__version__') else 'version not exposed')
print('Python:', platform.python_version())
print('Devices:', tf.config.list_physical_devices())
assert tuple(map(int,tf.__version__.split('.')[:2])) >= (2,16), 'This notebook targets TensorFlow >= 2.16 / Keras 3.'

## 2. Expected dataset structure

```text
data/homer_bart_2/
├── training_set/
│   ├── bart/  *.bmp, *.jpg, *.png
│   └── homer/ *.bmp, *.jpg, *.png
└── test_set/
    ├── bart/
    └── homer/
```

If your ZIP extracts to a different folder name, change `DATA_ROOT`. Never assume that a Kaggle archive is byte-for-byte identical to the historical Drive ZIP.

In [ ]:
from pathlib import Path
DATA_ROOT = Path('data/homer_bart_2')
TRAIN_DIR = DATA_ROOT/'training_set'
TEST_DIR = DATA_ROOT/'test_set'
for p in [TRAIN_DIR/'bart', TRAIN_DIR/'homer', TEST_DIR/'bart', TEST_DIR/'homer']:
    print(p, 'exists=',p.exists(), 'images=',sum(1 for f in p.glob('*') if f.suffix.lower() in {'.jpg','.jpeg','.png','.bmp'}))
assert TRAIN_DIR.exists() and TEST_DIR.exists(), 'Place/extract the archive using the documented folder layout.'

## 3. Data understanding before training

Inspect class counts, dimensions, corrupt files, duplicates and class mapping. Split training data into train/validation; retain the supplied test directory for final evaluation only.

In [ ]:
import collections
from PIL import Image
EXTS={'.jpg','.jpeg','.png','.bmp'}
def inspect_folder(root):
    rows=[]
    for label_dir in sorted(p for p in root.iterdir() if p.is_dir()):
        for path in label_dir.rglob('*'):
            if path.suffix.lower() in EXTS:
                try:
                    with Image.open(path) as im: rows.append((label_dir.name,path,im.size,im.mode))
                except Exception as e: print('Unreadable:',path,e)
    return rows
train_rows=inspect_folder(TRAIN_DIR); test_rows=inspect_folder(TEST_DIR)
print('Train:',collections.Counter(x[0] for x in train_rows))
print('Test:',collections.Counter(x[0] for x in test_rows))
print('Sample metadata:',train_rows[:5])

## 4. Build `tf.data` pipelines

For RGB images: decode, resize, cast to float32 and scale pixels by 255. The normalization equation is \(x' = x/255\), mapping integer pixels from [0,255] to [0,1]. Augmentation belongs only in the training pipeline.

In [ ]:
IMG_SIZE=(160,160); BATCH=32; SEED=42
train_ds=tf.keras.utils.image_dataset_from_directory(TRAIN_DIR,image_size=IMG_SIZE,batch_size=BATCH,validation_split=.2,subset='training',seed=SEED,label_mode='binary')
val_ds=tf.keras.utils.image_dataset_from_directory(TRAIN_DIR,image_size=IMG_SIZE,batch_size=BATCH,validation_split=.2,subset='validation',seed=SEED,label_mode='binary',shuffle=False)
test_ds=tf.keras.utils.image_dataset_from_directory(TEST_DIR,image_size=IMG_SIZE,batch_size=BATCH,label_mode='binary',shuffle=False)
class_names=train_ds.class_names
print('Class mapping:',dict(enumerate(class_names)))
normalise=lambda x,y:(tf.cast(x,tf.float32)/255.,y)
augment=tf.keras.Sequential([tf.keras.layers.RandomFlip('horizontal'),tf.keras.layers.RandomRotation(.04),tf.keras.layers.RandomZoom(.1)])
train_ds=train_ds.map(lambda x,y:(augment(tf.cast(x,tf.float32)/255.,training=True),y),num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)
val_ds=val_ds.map(normalise).cache().prefetch(tf.data.AUTOTUNE)
test_ds=test_ds.map(normalise).cache().prefetch(tf.data.AUTOTUNE)

## 5. CNN architecture and shape intuition

A convolution output dimension (valid padding) is \(H_{out}=\lfloor(H+2P-D(K-1)-1)/S+1\rfloor\). Pooling reduces spatial size; `GlobalAveragePooling2D` avoids a huge flattened dense layer.

In [ ]:
inputs=tf.keras.Input(shape=(*IMG_SIZE,3))
x=tf.keras.layers.Conv2D(32,3,padding='same',activation='relu')(inputs)
x=tf.keras.layers.MaxPooling2D()(x)
x=tf.keras.layers.Conv2D(64,3,padding='same',activation='relu')(x)
x=tf.keras.layers.MaxPooling2D()(x)
x=tf.keras.layers.Conv2D(128,3,padding='same',activation='relu')(x)
x=tf.keras.layers.GlobalAveragePooling2D()(x)
x=tf.keras.layers.Dropout(.3)(x)
outputs=tf.keras.layers.Dense(1,activation='sigmoid')(x)
model=tf.keras.Model(inputs,outputs)
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),loss='binary_crossentropy',metrics=['accuracy',tf.keras.metrics.Precision(),tf.keras.metrics.Recall()])
model.summary()

## 6. Train with validation, checkpoint and early stopping

Binary cross-entropy: \(L=-[y\log(p)+(1-y)\log(1-p)]\). Do not use the test set for epoch selection.

In [ ]:
callbacks=[tf.keras.callbacks.ModelCheckpoint('best_homer_bart.keras',monitor='val_loss',save_best_only=True),tf.keras.callbacks.EarlyStopping(monitor='val_loss',patience=5,restore_best_weights=True)]
history=model.fit(train_ds,validation_data=val_ds,epochs=30,callbacks=callbacks)

## 7. Final test evaluation and confusion matrix

In [ ]:
from sklearn.metrics import classification_report,confusion_matrix,ConfusionMatrixDisplay
import matplotlib.pyplot as plt
model=tf.keras.models.load_model('best_homer_bart.keras')
y_true=np.concatenate([y.numpy().ravel() for _,y in test_ds]).astype(int)
y_prob=model.predict(test_ds,verbose=0).ravel(); y_pred=(y_prob>=.5).astype(int)
print(classification_report(y_true,y_pred,target_names=class_names,digits=4))
ConfusionMatrixDisplay.from_predictions(y_true,y_pred,display_labels=class_names); plt.show()

## 8. Image treatment tips

- Keep aspect ratio when resizing if geometric distortion matters; use padding/letterboxing where appropriate.
- Use bilinear interpolation for RGB down/up-sampling; nearest-neighbour for categorical masks.
- Never apply random augmentation to validation/test data.
- Verify color channel order (PIL RGB vs OpenCV BGR).
- Check class balance and stratify splits where possible.
- For small datasets, report per-class precision/recall and macro-F1, not accuracy alone.
- Horizontal flips are not universally valid; confirm semantic invariance for the domain.

## Exercises
See `solutions/00_CNN_Image_Classification_Homer_Bart_TensorFlow_Solutions.ipynb`.